# 🧠📍 Smart Tagging

Metrica's **automatic** tactical annotation of the tactical video: phases of play and set
pieces, each one a time interval with a code and a set of tags.

This notebook covers loading it, what is in it, and the two things that will bite you if you
treat it as ground truth.

---

## Loading

The file is a **SportsCode XML**, the standard format in football performance analysis, so
[kloppy](https://kloppy.pysport.org/user-guide/loading-data/sportscode/) reads it directly.

`games.asset_paths()` builds the path, so nothing here spells a filename out and
`MTAH_DATA_DIR` keeps working if you store the bundle elsewhere.

In [12]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from kloppy import sportscode

import games

path = games.asset_paths()['smart_tagging']
events = sportscode.load(path).to_df()
print('%d instances' % len(events))
events.head(20)

638 instances


,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,NaN,NaN
1,2,1,0 days 00:33:05.240000,0 days 00:33:10.240000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN
2,3,1,0 days 00:41:20.480000,0 days 00:41:25.480000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN
3,4,1,0 days 00:50:18.280000,0 days 00:50:23.280000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN
4,5,1,0 days 01:15:46.400000,0 days 01:15:51.400000,SET PIECES,N/A,Kick Off (Start),2nd Half,None,NaN,NaN
5,6,1,0 days 01:48:40.000000,0 days 01:48:45.000000,SET PIECES,N/A,Kick Off (After Goal),2nd Half,None,NaN,NaN
6,7,1,0 days 01:17:41.720000,0 days 01:18:09.400000,PLAYERS IN THE BOX,Barcelona,NaN,2nd Half,4,NaN,NaN
7,8,1,0 days 01:18:42.760000,0 days 01:19:07.040000,PLAYERS IN THE BOX,Barcelona,NaN,2nd Half,4,NaN,NaN
8,9,1,0 days 01:21:23.200000,0 days 01:21:44.600000,PLAYERS IN THE BOX,Barcelona,NaN,2nd Half,2,NaN,NaN
9,10,1,0 days 01:22:46.080000,0 days 01:23:11.160000,PLAYERS IN THE BOX,Barcelona,NaN,2nd Half,2,NaN,NaN


In [13]:
events.tail(20)

,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry
618,619,1,0 days 01:32:01.680000,0 days 01:32:31.680000,DEFENDING IN DEFENSIVE THIRD,Atletic Club,NaN,2nd Half,None,NaN,NaN
619,620,1,0 days 01:43:52.920000,0 days 01:44:22.920000,DEFENDING IN DEFENSIVE THIRD,Barcelona,NaN,2nd Half,None,NaN,NaN
620,621,1,0 days 00:11:45.320000,0 days 00:12:15.320000,DEFENDING IN DEFENSIVE THIRD,Barcelona,NaN,1st Half,None,NaN,NaN
621,622,1,0 days 00:19:43.280000,0 days 00:20:13.280000,DEFENDING IN DEFENSIVE THIRD,Atletic Club,NaN,1st Half,None,NaN,NaN
622,623,1,0 days 01:30:09.800000,0 days 01:30:39.800000,DEFENDING IN DEFENSIVE THIRD,Barcelona,NaN,2nd Half,None,NaN,NaN
623,624,1,0 days 00:17:31.600000,0 days 00:18:01.600000,DEFENDING IN DEFENSIVE THIRD,Atletic Club,NaN,1st Half,None,NaN,NaN
624,625,1,0 days 01:54:01.600000,0 days 01:54:31.600000,DEFENDING IN DEFENSIVE THIRD,N/A,NaN,2nd Half,None,NaN,NaN
625,626,1,0 days 01:21:19.640000,0 days 01:21:49.640000,DEFENDING IN DEFENSIVE THIRD,Atletic Club,NaN,2nd Half,None,NaN,NaN
626,627,1,0 days 01:29:00.920000,0 days 01:29:30.920000,DEFENDING IN DEFENSIVE THIRD,Atletic Club,NaN,2nd Half,None,NaN,NaN
627,628,1,0 days 00:29:06.080000,0 days 00:29:36.080000,DEFENDING IN DEFENSIVE THIRD,Atletic Club,NaN,1st Half,None,NaN,NaN


## What is in it

Each row is one tagged interval:

- **timestamp / end_timestamp** — start and end, as pandas timedeltas from the first frame
- **code** — the phase or set piece, upper case (`SET PIECES`, `BUILD UP`, …)
- **Team** — which side the phase belongs to
- **Half** — `1st Half` or `2nd Half`, as strings
- plus per-code tags: `Type`, `Side`, `Max Players in the box`, `Direction of ball entry`

Note the capital in `Team`: kloppy passes the tag groups through with the names the XML uses, so
every tag column is title case. The full vocabulary is in [DATA.md](../DATA.md).

In [2]:
events['code'].value_counts()

code
PLAYERS IN THE BOX              97
BALL IN FINAL THIRD             93
BALL IN THE BOX                 72
PROGRESSION                     66
DEFENDING IN MIDDLE THIRD       66
SET PIECES                      59
CREATING CHANCES                30
DEFENDING IN DEFENSIVE THIRD    30
LONG BALL                       29
ATTACKING TRANSITION            24
DEFENSIVE TRANSITION            24
BUILD UP                        22
DEFENDING IN ATTACKING THIRD    22
GOALS                            4
Name: count, dtype: int64

### Codes are upper case, tag values are title case

`SET PIECES` and `Kick Off (Start)`. Match case-insensitively unless you enjoy silent empty
results.

In [3]:
set_pieces = events[events['code'].str.lower() == 'set pieces']
set_pieces['Type'].value_counts()

Type
Throw-in                 26
Corner Kick              12
Goal Kick                10
Free Kick                 5
Kick Off (After Goal)     4
Kick Off (Start)          2
Name: count, dtype: int64

## Converting to video frames

The timestamps are **seconds from the first frame of the tactical video**, so a video frame —
and therefore a submission row — is just `round(seconds * 25)`. There is no offset to discover.

`starter/load_smart_tagging.py` does this without kloppy, and re-checks the time base against
both kickoffs every time it runs.

In [4]:
FPS = games.GAMES[games.GAME]['fps']

frames = pd.DataFrame({
    'code': events['code'],
    'Team': events['Team'],
    'start_frame': (events['timestamp'].dt.total_seconds() * FPS).round().astype(int),
    'end_frame': (events['end_timestamp'].dt.total_seconds() * FPS).round().astype(int),
})
frames.head()

,code,Team,start_frame,end_frame
0,SET PIECES,N/A,11097,11222
1,SET PIECES,N/A,49631,49756
2,SET PIECES,N/A,62012,62137
3,SET PIECES,N/A,75457,75582
4,SET PIECES,N/A,113660,113785


## Two things to be careful about

### 1. `Team` is not always a team

36 instances are tagged `N/A` — including **every throw-in**, which are exactly the restarts
where knowing the side would be worth the most. Handle it explicitly; do not let it fall
through to a default.

In [5]:
events['Team'].value_counts(dropna=False)

Team
Barcelona       314
Atletic Club    288
N/A              36
Name: count, dtype: int64

### 2. An interval is a window, not an instant

A set piece is tagged as a span of a few seconds *around* the restart, not at the frame the ball
is struck. The two `Kick Off (Start)` tags below both **contain** the real kickoff rather than
starting on it — they open about 3.5 seconds early.

So `start_frame` is not the restart frame, and the same imprecision applies to every other
interval in the file. Use a tag to say *roughly when* something happened, and the tracking to say
exactly when.

In [6]:
kickoffs = set_pieces[set_pieces['Type'].str.lower() == 'kick off (start)'].copy()
kickoffs['start_frame'] = (kickoffs['timestamp'].dt.total_seconds() * FPS).round().astype(int)
kickoffs['end_frame'] = (kickoffs['end_timestamp'].dt.total_seconds() * FPS).round().astype(int)
kickoffs['real_kickoff'] = [11185, 113742]          # from the ground truth
kickoffs['contains_it'] = ((kickoffs['start_frame'] <= kickoffs['real_kickoff'])
                           & (kickoffs['real_kickoff'] <= kickoffs['end_frame']))

kickoffs[['Half', 'start_frame', 'end_frame', 'real_kickoff', 'contains_it']]

,Half,start_frame,end_frame,real_kickoff,contains_it
0,1st Half,11097,11222,11185,True
4,2nd Half,113660,113785,113742,True


---

## Why this is worth using anyway

Every phase carries a **team** tag, and every possession has a team. Every set piece is a
restart, and every restart ends a dead-ball stretch. The correspondence is not exact — that is
the challenge — but it is a strong prior, for free, on both halves of the label you are asked
to produce.

Just remember the intervals are *phases*, not possessions: a Build Up can span one possession,
stop short of one, or cover two.

**Next:** [`02_tracking_atd.ipynb`](02_tracking_atd.ipynb) for the tracking data, then
[`03_first_submission.ipynb`](03_first_submission.ipynb) to build and score a submission.